# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Luis Arturo Sánchez Davila
#### **Matrícula:** A01840576

## Objetivo
Notebook de **carga** del proyecto MiBici: obtención del dataset de viajes con `dataset.py` y su descripción (Paso 1). El EDA, la limpieza y el modelado de P3 están en notebooks aparte (ver al final).

La pregunta que me corresponde es la **P3 (clustering)**: ¿qué grupos de estaciones comparten patrones de uso y dónde se ubican?

La lógica que se reutilice se trasladará al módulo del proyecto; en este notebook se desarrolla y se documenta la evidencia que la justifica.

## Dataset
**Datos abiertos de MiBici** (sistema de bicicleta pública del Área Metropolitana de Guadalajara), tal como los deja `tc5061_proyecto_equipo07/dataset.py`:
- **`data/interim/viajes_AAAA-MM_AAAA-MM.parquet`:** todos los meses del rango en un solo Parquet normalizado (nombres, fechas y tipos), sin limpiar. Es la entrada del EDA.
- **`data/processed/viajes_limpios_AAAA-MM_AAAA-MM.parquet`:** el mismo dataset después de las reglas de limpieza comunes (sección 3.1 de `0.06-lasd-limpieza-p3`).
- Cada registro es un viaje: `viaje_id`, `usuario_id`, `genero`, `anio_de_nacimiento`, `inicio_del_viaje`, `fin_del_viaje`, `origen_id` y `destino_id`; la ingesta agrega `periodo_archivo`.
- Fuente original: [Datos abiertos de MiBici](https://mibici.net/es/datos-abiertos/)
- Rango analizado: **enero de 2020 a septiembre de 2026**, acordado por el equipo para homologar el EDA. El portal retiró 2023-04 y 2023-05 el 14-oct-2026 y los volvió a publicar el 15-oct-2026 (ver `0.02-dsh-exploracion-carga`).

## Nota de reproducibilidad
El notebook se ejecuta desde la carpeta `notebooks/` con el entorno descrito en el README. Los CSV de `data/raw/` y los Parquet no se versionan en Git; lo versionado es el manifiesto `references/fuentes-datos.csv` (registros y SHA-256 por archivo). Los datos se generan con `make data` (`python -m tc5061_proyecto_equipo07.dataset todo`), que descarga lo que falte, verifica cada CSV contra el manifiesto y no se detiene en silencio si un mes no cuadra.

**Cifras:** las salidas y observaciones de las secciones 1 a 4 corresponden a la corrida del 15-oct-2026 con los **81 meses** del rango (27,086,557 viajes; 27,026,645 tras la limpieza). Una versión anterior de este notebook se escribió con 79 meses, porque el portal había retirado 2023-04 y 2023-05; sus cifras ya no aplican.

In [1]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pyarrow

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)

Python      : 3.11.9
pandas      : 3.0.5
numpy       : 1.26.4
pyarrow     : 25.0.1


## Configuración global

Las rutas y el rango de meses salen de `tc5061_proyecto_equipo07/config.py`, para que todos los notebooks y `dataset.py` usen los mismos valores. Pandas muestra todas las columnas.

In [2]:
# Configuración compartida (el notebook se ejecuta desde notebooks/)
try:
    from tc5061_proyecto_equipo07 import dataset
except ModuleNotFoundError:  # paquete sin instalar; `make requirements` lo instala con `-e .`
    sys.path.insert(0, str(Path.cwd().parent))
    from tc5061_proyecto_equipo07 import dataset

from tc5061_proyecto_equipo07.config import (
    MES_FIN,
    MES_INICIO,
    RUTA_MANIFIESTO,
    RUTA_VIAJES_INTERIM,
)

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

2026-10-15 21:36:18.384 | INFO     | tc5061_proyecto_equipo07.config:<module>:14 - PROJ_ROOT path is: /Users/lsanchez/Maestria_proyectos/MLOps-ProyectoMiBici-Equipo7


## Paso 1: Carga y descripción del dataset

En este paso se obtienen los Parquet mensuales, se leen y se valida que la carga sea completa antes de describir el dataset. Cualquier diferencia se documenta como hallazgo y se resuelve con evidencia, no con supuestos.

### 1.1 Obtención de los datos con `dataset.py`

La ingesta ya no vive en el notebook. `tc5061_proyecto_equipo07/dataset.py` la ejecuta en tres etapas (cada una es un comando; `todo` las corre en orden):

| Etapa | Entrada | Salida |
|---|---|---|
| `descargar` | Portal de MiBici | CSV en `data/raw/` y manifiesto `references/fuentes-datos.csv` (registros y SHA-256) |
| `construir` | CSV de `data/raw/` | `data/interim/viajes_AAAA-MM_AAAA-MM.parquet`, normalizado y validado |
| `limpiar` | Parquet de `data/interim/` | `data/processed/viajes_limpios_AAAA-MM_AAAA-MM.parquet` |

La codificación (UTF-8 o latin-1), el encabezado dañado de 2024-02 y 2024-05, y los dos formatos de fecha se resuelven en `dataset.py`; se detiene si un encabezado es desconocido, si un valor no se convierte, si un viaje inicia fuera del mes de su archivo o si un CSV no coincide con el manifiesto. Esta celda solo verifica que los datos existan y los compara contra el manifiesto.

In [3]:
# Los datos los produce dataset.py: descargar -> construir -> limpiar (make data)
EJECUTAR_INGESTA = False  # True: corre dataset.todo desde aquí (descarga lo que falte; puede tardar)

if EJECUTAR_INGESTA:
    dataset.todo(MES_INICIO, MES_FIN)

if not RUTA_VIAJES_INTERIM.exists():
    raise FileNotFoundError(
        f"Falta {RUTA_VIAJES_INTERIM.name}; ejecutar `make data` o poner EJECUTAR_INGESTA = True"
    )

# Lo que debe haber en el rango, según el manifiesto versionado en Git
manifiesto = pd.read_csv(RUTA_MANIFIESTO)
manifiesto = manifiesto[manifiesto["archivo"].str.startswith("datos_abiertos")
                        & manifiesto["periodo"].between(MES_INICIO, MES_FIN)]
print("Archivos de viajes en el manifiesto:", len(manifiesto))
print("Registros esperados                :", f"{int(manifiesto['registros'].sum()):,}")
print("Versión de los datos               :", dataset.version_datos())

Archivos de viajes en el manifiesto: 81
Registros esperados                : 27,086,557
Versión de los datos               : c3b84e1ab511


**Observaciones de la obtención:**
- **Origen:** los datos los genera `dataset.py`; el notebook ya no descarga ni convierte nada, de modo que todos los notebooks parten del mismo archivo.
- **Trazabilidad:** el manifiesto (`references/fuentes-datos.csv`) guarda el número de registros y el SHA-256 de cada CSV, y la versión de los datos es el hash de esos hashes. Si un mes cambia en el portal, cambia la versión y se ve en el diff de Git.
- **Meses sin datos:** `MESES_SIN_PUBLICAR` (en `config.py`) solo tolera 2023-04 y 2023-05 si el portal vuelve a retirarlos; cualquier otro hueco detiene la descarga.
- **Sustituye:** la conversión CSV a Parquet que antes estaba en esta sección (y que no estaba versionada en el paquete) y la comparación contra los Parquet de `0.01`.

In [4]:
# Lectura del Parquet normalizado (viajes sin limpiar) y validación contra el manifiesto
viajes = pd.read_parquet(RUTA_VIAJES_INTERIM)

esperados = int(manifiesto["registros"].sum())
assert len(viajes) == esperados, (
    f"El Parquet tiene {len(viajes):,} registros y el manifiesto indica {esperados:,}; "
    "ejecutar `dataset construir`"
)

# Los meses sin datos se calculan de los datos, no de una lista fija
periodos_en_datos = set(viajes["periodo_archivo"].astype(str).unique())
MESES_SIN_DATOS = [str(p) for p in pd.period_range(MES_INICIO, MES_FIN, freq="M")
                   if str(p) not in periodos_en_datos]

print("Archivos leídos:", len(periodos_en_datos))
print(f"Registros      : {len(viajes):,}")
print("Columnas       :", viajes.shape[1])
print("Meses sin datos:", MESES_SIN_DATOS or "ninguno")

Archivos leídos: 81
Registros      : 27,086,557
Columnas       : 9
Meses sin datos: ninguno


**Observaciones de la lectura:**
- **Fuente:** `data/interim/viajes_2020-01_2026-09.parquet`, generado por `dataset.construir`, que verificó el SHA-256 de cada CSV contra el manifiesto y que no se perdieran registros.
- **Validación:** el número de registros leídos coincide con el del manifiesto; la celda se detiene si no.
- **Meses:** los meses del rango sin archivo se calculan de los datos (`MESES_SIN_DATOS`). Con los 81 meses publicados no hay ninguno; si el portal vuelve a retirar 2023-04 y 2023-05, aparecen aquí y se excluyen de la tabla de demanda.
- **Cifras:** el total de viajes, el mes con menos viajes (2020-04) y el de más viajes (2026-09) se leen de las salidas de arriba y de la sección 2.3 de `0.05-lasd-eda-p3`.

### 1.3 Descripción del dataset

Se revisan los tipos, los nulos y el rango de fechas para saber qué columnas requieren limpieza.

In [5]:
# Columnas, tipos, valores no nulos y memoria
viajes.info(show_counts=True, memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 27086557 entries, 0 to 27086556
Data columns (total 9 columns):
 #   Column              Non-Null Count     Dtype         
---  ------              --------------     -----         
 0   viaje_id            27086557 non-null  Int64         
 1   usuario_id          27086557 non-null  Int64         
 2   genero              26948912 non-null  category      
 3   anio_de_nacimiento  27051936 non-null  Int16         
 4   inicio_del_viaje    27086557 non-null  datetime64[us]
 5   fin_del_viaje       27086557 non-null  datetime64[us]
 6   origen_id           27086557 non-null  Int32         
 7   destino_id          27086557 non-null  Int32         
 8   periodo_archivo     27086557 non-null  category      
dtypes: Int16(1), Int32(2), Int64(2), category(2), datetime64[us](2)
memory usage: 1.2 GB


In [6]:
# Nulos y valores distintos por columna
resumen = pd.DataFrame({
    "nulos": viajes.isna().sum(),
    "% nulos": (viajes.isna().mean() * 100).round(2),
    "distintos": viajes.nunique(),
})
resumen

,nulos,% nulos,distintos
viaje_id,0,0.00,27085799
usuario_id,0,0.00,81245
genero,137645,0.51,2
anio_de_nacimiento,34621,0.13,87
inicio_del_viaje,0,0.00,23238062
fin_del_viaje,0,0.00,23190194
origen_id,0,0.00,475
destino_id,0,0.00,475
periodo_archivo,0,0.00,81


In [7]:
# Rango de fechas de inicio y fin de los viajes
print("Inicio:", viajes["inicio_del_viaje"].min(), "->", viajes["inicio_del_viaje"].max())
print("Fin   :", viajes["fin_del_viaje"].min(), "->", viajes["fin_del_viaje"].max())

Inicio: 2020-01-01 06:02:20 -> 2026-09-30 23:59:48
Fin   : 2020-01-01 06:05:38 -> 2026-10-01 00:14:52


**Observaciones de la descripción del dataset:**
- **Estructura:** 27,086,557 registros y 9 columnas (las 8 originales más `periodo_archivo`). `genero` y `periodo_archivo` son categorías; los identificadores, el año de nacimiento y las estaciones son enteros que admiten nulos.
- **Nulos:** solo `genero` (137,645, 0.51 %) y `anio_de_nacimiento` (34,621, 0.13 %) tienen valores nulos; las otras siete columnas están completas, incluidas las fechas.
- **Identificadores repetidos:** 27,085,799 `viaje_id` distintos para 27,086,557 registros, es decir, 758 repetidos; en 2.2 (`0.05-lasd-eda-p3`) se confirma que son filas idénticas.
- **Fechas:** los viajes inician entre 2020-01-01 06:02:20 y 2026-09-30 23:59:48; el fin llega hasta 2026-10-01 00:14:52, porque los viajes iniciados al final de septiembre terminan en octubre.
- **Limitación:** el rango de fechas confirma el periodo pero no la continuidad: los meses con datos se ven en la sección 2.3 de `0.05-lasd-eda-p3`.

## Siguiente paso

## Serie de notebooks de P3
| Notebook | Contenido | Entrega |
|---|---|---|
| `0.03-lasd-exploracion-carga` | Paso 1: carga y descripción | Dataset normalizado en `data/interim/` |
| `0.05-lasd-eda-p3` | Paso 2: análisis exploratorio | Hallazgos de calidad, tiempo y estaciones |
| `0.06-lasd-limpieza-p3` | Paso 3: limpieza, partición y preprocesamiento | `data/processed/` (viajes de modelado, estaciones y demanda) |
| `0.07-lasd-baseline-p3` | Paso 4 (4.1 a 4.4): modelo base y registro en MLflow | Variables por estación y métricas del baseline |
| `0.08-lasd-mejoramodelo-p3` | Paso 4 (4.5 a 4.9): modelo alternativo y registro en MLflow | Comparación contra el baseline |